# 03 - Run Hybrid Models
Train hybrid quantum-classical students (NoKD and KD).

**Fix (2026-03-28):** `quantum_layer.py` now uses a manual batch loop
instead of `TorchLayer`, which avoids the `shape '[32,-1]' is invalid`
RuntimeError in PennyLane 0.38+.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Reload modules in case of cached imports
import importlib
import src.models.quantum_layer
import src.models.student_hybrid
importlib.reload(src.models.quantum_layer)
importlib.reload(src.models.student_hybrid)

from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = os.path.abspath('../data/raw/wdbc.csv')
print('CSV path:', CSV)
print('File exists:', os.path.exists(CSV))

## Step 1 - Re-run Teacher (needed for KD logits)

In [ ]:
teacher = train_teacher_cv(CSV, pca_dim=4, n_splits=5, seed=42)
print('Teacher training complete.')

## Step 2 - Student Hybrid NoKD
Quantum layer position: **middle**. No distillation.

In [ ]:
metrics_nokd = run_student_cv(
    CSV,
    teacher_fold_outputs=None,
    model_type='hybrid',
    use_kd=False,
    quantum_position='middle',
    pca_dim=4,
    n_qubits=4,
    n_splits=5,
    seed=42,
    epochs=60,
    lr=1e-3,
    batch_size=16,
    exp_name='student_hybrid_nokd'
)
print('NoKD done.')

## Step 3 - Student Hybrid KD
Same architecture, with teacher soft-label distillation (T=2, alpha=0.5).

In [ ]:
metrics_kd = run_student_cv(
    CSV,
    teacher_fold_outputs=teacher,
    model_type='hybrid',
    use_kd=True,
    alpha=0.5,
    T=2.0,
    quantum_position='middle',
    pca_dim=4,
    n_qubits=4,
    n_splits=5,
    seed=42,
    epochs=60,
    lr=1e-3,
    batch_size=16,
    exp_name='student_hybrid_kd'
)
print('KD done.')

## Results Summary

In [ ]:
import numpy as np

def print_summary(name, metrics):
    for k in ['auc', 'f1', 'mcc', 'acc']:
        vals = [m[k] for m in metrics]
        print(f'[{name}] {k}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')
    times = [m['train_time'] for m in metrics]
    print(f'[{name}] train_time: {np.mean(times):.1f}s avg per fold')

print_summary('Hybrid-NoKD', metrics_nokd)
print()
print_summary('Hybrid-KD',   metrics_kd)